# Framingham Heart Study: 10-Year CHD Prediction

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning pipeline for predicting the 10-year risk of Coronary Heart Disease (CHD) based on the Framingham Heart Study dataset. The Framingham Heart Study is a longitudinal cohort study that began in 1948 and is dedicated to identifying common factors or characteristics that contribute to cardiovascular disease (CVD).

Coronary Heart Disease (CHD) is a serious global health concern, and early prediction can enable timely interventions and lifestyle modifications, significantly improving patient outcomes. This project aims to build a robust classification model to assess the risk of developing CHD within ten years.

The dataset contains various demographic, behavioral, and medical information for each patient. The target variable is `TenYearCHD`, a binary indicator where `0` means no CHD within 10 years and `1` means CHD development within 10 years.

This notebook will cover the following stages:
1.  **Data Loading**: Importing the dataset and initial inspection.
2.  **Exploratory Data Analysis (EDA)**: Understanding data distributions, missing values, and initial relationships.
3.  **Data Preprocessing**: Handling missing values, outliers, and feature scaling.
4.  **Visualizations**: In-depth plots using Plotly to represent EDA insights, correlations, and model results.
5.  **Feature Engineering & Selection**: Creating new features and selecting the most relevant ones.
6.  **Model Training**: Implementing various classification algorithms.
7.  **Model Evaluation**: Assessing model performance using appropriate metrics.
8.  **Optimization Concepts**: Discussing local vs. global minima and residuals.
9.  **Overfitting/Underfitting Analysis**: Identifying and addressing common modeling pitfalls.
10. **Hyperparameter Tuning**: Optimizing model parameters for better performance.
11. **Prediction on Example Data**: Demonstrating how to use the final model for new predictions.
12. **Final Model Selection & Saving**: Choosing the best model and persisting it.
13. **Insights & Conclusion**: Summarizing findings and suggesting future work.

Let's begin by setting up our environment and loading the data.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
import pickle
import os
import logging
import sys

# --- Configuration and Setup ---

# Create directories for logs and artifacts if they don't exist
log_dir = 'ml_logs'
artifacts_dir = 'artifacts'
os.makedirs(log_dir, exist_ok=True)
os.makedirs(artifacts_dir, exist_ok=True)

# Configure logging
log_file_path = os.path.join(log_dir, 'ml_pipeline.log')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(log_file_path), logging.StreamHandler(sys.stdout)])

logging.info("ML pipeline started.")

# Define the data path (assuming 'framingham.csv' is in a 'data' directory)
# Please ensure you have 'framingham.csv' in the 'data' directory.
data_path = 'data/framingham.csv'

# Set a random seed for reproducibility
np.random.seed(42)


## 2. Data Loading

In this section, we will load the dataset from the specified CSV file into a pandas DataFrame. We'll also perform an initial check to ensure the data has been loaded correctly and log any potential issues during this process.


In [ ]:
# Function to load data with error handling
def load_data(path):
    try:
        df = pd.read_csv(path)
        logging.info(f"Dataset loaded successfully from {path}. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        logging.error(f"Error: The file '{path}' was not found. Please ensure the CSV file is in the correct directory.")
        sys.exit(1) # Exit if data file is not found
    except Exception as e:
        logging.error(f"An unexpected error occurred during data loading: {e}")
        sys.exit(1)

# Load the dataset
df = load_data(data_path)

# Display the first few rows of the DataFrame
logging.info("Displaying the first 5 rows of the dataset:")
print(df.head())

# Display basic information about the dataset
logging.info("Displaying basic info of the dataset:")
df.info()


## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is a crucial step to understand the characteristics of our dataset. We will examine:
*   **Descriptive Statistics**: Summary statistics for numerical features.
*   **Missing Values**: Identify and quantify missing data points.
*   **Duplicate Rows**: Check for and handle any duplicate entries.
*   **Data Types**: Verify that data types align with our expectations.
*   **Target Variable Distribution**: Understand the balance of our target variable (`TenYearCHD`).


In [ ]:
logging.info("Starting Exploratory Data Analysis (EDA).")

# 1. Descriptive Statistics
logging.info("Displaying descriptive statistics for numerical features:")
print(df.describe().T) # Transpose for better readability

# 2. Check for Missing Values
logging.info("Checking for missing values in each column:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': missing_percentage})
print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

if missing_df['Missing Count'].sum() == 0:
    logging.info("No missing values found in the dataset.")
else:
    logging.warning(f"Missing values found in the following columns: {list(missing_df[missing_df['Missing Count'] > 0].index)}. These will be handled during preprocessing.")

# 3. Check for Duplicate Rows
logging.info("Checking for duplicate rows:")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    logging.warning(f"Found {duplicate_rows} duplicate rows. Removing them.")
    df.drop_duplicates(inplace=True)
    logging.info(f"Duplicate rows removed. New dataset shape: {df.shape}")
else:
    logging.info("No duplicate rows found.")

# 4. Check Data Types (already done by df.info(), but a quick reiteration)
logging.info("Data types of columns:")
print(df.dtypes)

# 5. Target Variable Distribution
logging.info("Analyzing target variable (TenYearCHD) distribution:")
target_distribution = df['TenYearCHD'].value_counts(normalize=True) * 100
print(target_distribution)

if target_distribution[0] > 70 or target_distribution[1] > 70: # A simple heuristic for imbalance
    logging.warning(f"The target variable 'TenYearCHD' is imbalanced. Class 0: {target_distribution[0]:.2f}%, Class 1: {target_distribution[1]:.2f}%. This will be addressed during modeling.")
else:
    logging.info("Target variable distribution appears relatively balanced.")

logging.info("EDA completed.")

# Store original columns before preprocessing for later reference if needed
original_columns = df.columns.tolist()
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist() # Should be empty for this dataset

# Given schema: {'male': 'int64', 'age': 'int64', 'education': 'float64', 'currentSmoker': 'int64', 'cigsPerDay': 'float64', 'BPMeds': 'float64', 'prevalentStroke': 'int64', 'prevalentHyp': 'int64', 'diabetes': 'int64', 'totChol': 'float64', 'sysBP': 'float64', 'diaBP': 'float64', 'BMI': 'float64', 'heartRate': 'float64', 'glucose': 'float64', 'TenYearCHD': 'int64'}
# Based on schema, all are numerical, but some are binary/ordinal categorical treated as numerical.
binary_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']
ordinal_cols = ['education']
continuous_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']


## 4. Preprocessing

This section focuses on preparing the data for model training. We will perform the following steps:
1.  **Handling Missing Values**: Impute missing values based on suitable strategies.
2.  **Outlier Handling**: Identify and treat outliers to prevent them from skewing our model.
3.  **Feature Engineering**: Create new features to potentially improve model performance.
4.  **Feature Scaling**: Standardize numerical features for algorithms sensitive to scale.


In [ ]:
logging.info("Starting Data Preprocessing.")

# Make a copy of the DataFrame to preserve the original if needed
df_processed = df.copy()

# --- 1. Handling Missing Values ---
logging.info("Handling missing values.")

# Identify columns with missing values
missing_cols = df_processed.columns[df_processed.isnull().any()].tolist()
logging.info(f"Columns with missing values: {missing_cols}")

# Imputation strategy:
# For 'education', 'cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose', 'sysBP', 'diaBP' - use median imputation as it's robust to outliers.
# For 'BPMeds', which is binary (0/1) but represented as float, mode imputation is appropriate (assuming NaN means 'no').
imputer_median_cols = ['education', 'cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose', 'sysBP', 'diaBP']
imputer_mode_cols = ['BPMeds'] # Although BPMeds can have missing values, 0 is the mode, implying no BP medication.

for col in imputer_median_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        logging.info(f"Missing values in '{col}' imputed with median: {median_val}")

for col in imputer_mode_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        mode_val = df_processed[col].mode()[0] # .mode() can return multiple values if multimodal, so take first
        df_processed[col].fillna(mode_val, inplace=True)
        logging.info(f"Missing values in '{col}' imputed with mode: {mode_val}")

# Verify no more missing values
if df_processed.isnull().sum().sum() == 0:
    logging.info("All missing values handled successfully.")
else:
    logging.error("Failed to handle all missing values. Rechecking for remaining NaNs.")
    print(df_processed.isnull().sum()[df_processed.isnull().sum() > 0])

# --- 2. Outlier Handling ---
logging.info("Handling outliers using IQR method for continuous numerical features.")

# Define a function to cap outliers
def cap_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap values
    initial_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    if initial_outliers > 0:
        df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
        df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
        logging.info(f"Capped {initial_outliers} outliers in '{col}' using IQR method. Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
    return df

# Apply outlier capping to continuous numerical columns
# Exclude binary and ordinal features from outlier capping as they represent categories
numerical_for_outliers = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

for col in numerical_for_outliers:
    try:
        if col in df_processed.columns:
            df_processed = cap_outliers_iqr(df_processed, col)
        else:
            logging.warning(f"Column '{col}' not found for outlier capping.")
    except Exception as e:
        logging.error(f"Error during outlier capping for column '{col}': {e}")


# --- 3. Feature Engineering ---
logging.info("Starting Feature Engineering.")

# Create a new feature for 'BMI_Category'
# Based on WHO classifications:
# <18.5 = Underweight
# 18.5-24.9 = Normal weight
# 25-29.9 = Overweight
# >=30 = Obese
def get_bmi_category(bmi):
    if bmi < 18.5:
        return 0 # Underweight
    elif 18.5 <= bmi < 25:
        return 1 # Normal weight
    elif 25 <= bmi < 30:
        return 2 # Overweight
    else:
        return 3 # Obese

if 'BMI' in df_processed.columns:
    df_processed['BMI_Category'] = df_processed['BMI'].apply(get_bmi_category)
    logging.info("Created 'BMI_Category' feature.")
else:
    logging.warning("BMI column not found for 'BMI_Category' feature engineering.")

# Create a new feature for 'BP_Category' (Blood Pressure Category)
# Based on common BP classifications (e.g., AHA/ACC 2017 guidelines):
# Normal: <120/<80
# Elevated: 120-129/<80
# High BP (Hypertension Stage 1): 130-139 or 80-89
# High BP (Hypertension Stage 2): >=140 or >=90
# Hypertensive Crisis: >180 and/or >120
def get_bp_category(sysBP, diaBP):
    if sysBP < 120 and diaBP < 80:
        return 0 # Normal
    elif (120 <= sysBP <= 129) and diaBP < 80:
        return 1 # Elevated
    elif (130 <= sysBP <= 139) or (80 <= diaBP <= 89):
        return 2 # Hypertension Stage 1
    elif (sysBP >= 140) or (diaBP >= 90):
        return 3 # Hypertension Stage 2
    else: # Fallback for edge cases, though categories should cover most.
        return 0

if 'sysBP' in df_processed.columns and 'diaBP' in df_processed.columns:
    df_processed['BP_Category'] = df_processed.apply(lambda row: get_bp_category(row['sysBP'], row['diaBP']), axis=1)
    logging.info("Created 'BP_Category' feature.")
else:
    logging.warning("sysBP or diaBP columns not found for 'BP_Category' feature engineering.")


# Create a new feature for 'Cholesterol_Risk' (High Cholesterol)
# Generally, Total Cholesterol < 200 mg/dL is desirable, 200-239 is borderline high, >= 240 is high.
if 'totChol' in df_processed.columns:
    df_processed['HighCholesterol'] = (df_processed['totChol'] >= 240).astype(int)
    logging.info("Created 'HighCholesterol' feature.")
else:
    logging.warning("totChol column not found for 'HighCholesterol' feature engineering.")

# --- 4. Feature Scaling ---
logging.info("Starting Feature Scaling.")

# Identify columns to scale (continuous numerical features and education)
# Exclude binary features, the target variable, and the newly created categorical features
features_to_scale = [col for col in continuous_cols + ['education'] if col in df_processed.columns]

# Initialize the StandardScaler
scaler = StandardScaler()

try:
    # Fit and transform the identified features
    df_processed[features_to_scale] = scaler.fit_transform(df_processed[features_to_scale])
    logging.info(f"Features {features_to_scale} scaled using StandardScaler.")
except Exception as e:
    logging.error(f"Error during feature scaling: {e}")

logging.info("Data Preprocessing completed.")
print("Processed DataFrame head after all steps:")
print(df_processed.head())
print("Processed DataFrame info after all steps:")
df_processed.info()


## 5. Visual Representation of EDA

This section utilizes Plotly to create interactive and informative visualizations that help us further understand the dataset characteristics and relationships between features.


In [ ]:
logging.info("Generating visual representations of EDA using Plotly.")

# --- 1. Target Variable Distribution ---
fig = px.pie(df_processed, names='TenYearCHD', title='Distribution of TenYearCHD (Target Variable)',
             labels={'TenYearCHD': 'CHD Risk'}, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()
logging.info("Plotly: TenYearCHD distribution pie chart generated.")

# --- 2. Histograms for Numerical Features ---
# Use the continuous_cols list (which was from original, make sure to add engineered if they are continuous and scaled)
numerical_features_for_hist = [col for col in continuous_cols if col in df_processed.columns] # Using original continuous cols as a base

fig = make_subplots(rows=len(numerical_features_for_hist)//2 + len(numerical_features_for_hist)%2, cols=2,
                    subplot_titles=[f'Distribution of {col}' for col in numerical_features_for_hist])
for i, col in enumerate(numerical_features_for_hist):
    row = i // 2 + 1
    col_num = i % 2 + 1
    fig.add_trace(go.Histogram(x=df_processed[col], name=col, marker_color=px.colors.qualitative.Set2[i%8]),
                  row=row, col=col_num)
fig.update_layout(height=400 * (len(numerical_features_for_hist)//2 + len(numerical_features_for_hist)%2),
                  title_text="Histograms of Numerical Features", showlegend=False)
fig.show()
logging.info("Plotly: Histograms of numerical features generated.")

# --- 3. Box Plots for Outlier Visualization (before capping for demonstration, or after to show effect) ---
# Let's show it on the *processed* data, indicating the effect of capping.
fig = make_subplots(rows=len(numerical_for_outliers)//2 + len(numerical_for_outliers)%2, cols=2,
                    subplot_titles=[f'Box Plot of {col}' for col in numerical_for_outliers])
for i, col in enumerate(numerical_for_outliers):
    row = i // 2 + 1
    col_num = i % 2 + 1
    fig.add_trace(go.Box(y=df_processed[col], name=col, marker_color=px.colors.qualitative.Set1[i%9]),
                  row=row, col=col_num)
fig.update_layout(height=400 * (len(numerical_for_outliers)//2 + len(numerical_for_outliers)%2),
                  title_text="Box Plots of Numerical Features (After Outlier Capping)", showlegend=False)
fig.show()
logging.info("Plotly: Box plots of numerical features generated (after capping).")

# --- 4. Count Plots for Binary/Ordinal and Engineered Categorical Features ---
categorical_for_counts = binary_cols + ordinal_cols + ['BMI_Category', 'BP_Category', 'HighCholesterol']
categorical_for_counts = [col for col in categorical_for_counts if col in df_processed.columns]

if categorical_for_counts:
    fig = make_subplots(rows=len(categorical_for_counts)//2 + len(categorical_for_counts)%2, cols=2,
                        subplot_titles=[f'Count of {col}' for col in categorical_for_counts])
    for i, col in enumerate(categorical_for_counts):
        row = i // 2 + 1
        col_num = i % 2 + 1
        counts = df_processed[col].value_counts().reset_index()
        counts.columns = [col, 'count']
        fig.add_trace(go.Bar(x=counts[col].astype(str), y=counts['count'], name=col, marker_color=px.colors.qualitative.T10[i%10]),
                      row=row, col=col_num)
    fig.update_layout(height=400 * (len(categorical_for_counts)//2 + len(categorical_for_counts)%2),
                      title_text="Count Plots of Categorical and Binary Features", showlegend=False)
    fig.show()
    logging.info("Plotly: Count plots of categorical/binary features generated.")
else:
    logging.info("No categorical features identified for count plots.")

# --- 5. Relationship with Target Variable ---
# For a few key numerical features, visualize their distribution across CHD status
key_numerical_features = ['age', 'sysBP', 'totChol', 'glucose', 'BMI', 'cigsPerDay']
key_numerical_features = [col for col in key_numerical_features if col in df_processed.columns]

if key_numerical_features:
    fig = make_subplots(rows=len(key_numerical_features)//2 + len(key_numerical_features)%2, cols=2,
                        subplot_titles=[f'{col} by TenYearCHD' for col in key_numerical_features])
    for i, col in enumerate(key_numerical_features):
        row = i // 2 + 1
        col_num = i % 2 + 1
        fig.add_trace(go.Box(y=df_processed[col][df_processed['TenYearCHD']==0], name=f'{col} (No CHD)',
                             marker_color='blue', showlegend=True, boxpoints='outliers'),
                      row=row, col=col_num)
        fig.add_trace(go.Box(y=df_processed[col][df_processed['TenYearCHD']==1], name=f'{col} (CHD)',
                             marker_color='red', showlegend=True, boxpoints='outliers'),
                      row=row, col=col_num)
    fig.update_layout(height=400 * (len(key_numerical_features)//2 + len(key_numerical_features)%2),
                      title_text="Numerical Feature Distribution by TenYearCHD Status", showlegend=True)
    fig.show()
    logging.info("Plotly: Box plots showing numerical feature distribution by target generated.")
else:
    logging.warning("No key numerical features identified for target relationship plots.")


# For binary/categorical features, visualize their proportion across CHD status
key_categorical_features = ['male', 'currentSmoker', 'prevalentHyp', 'diabetes', 'BMI_Category', 'BP_Category', 'HighCholesterol']
key_categorical_features = [col for col in key_categorical_features if col in df_processed.columns]

if key_categorical_features:
    fig = make_subplots(rows=len(key_categorical_features)//2 + len(key_categorical_features)%2, cols=2,
                        subplot_titles=[f'{col} by TenYearCHD' for col in key_categorical_features])
    for i, col in enumerate(key_categorical_features):
        row = i // 2 + 1
        col_num = i % 2 + 1
        crosstab_df = pd.crosstab(df_processed[col], df_processed['TenYearCHD'], normalize='index').reset_index()
        crosstab_df.columns = [col, 'No CHD', 'CHD']
        fig.add_trace(go.Bar(x=crosstab_df[col].astype(str), y=crosstab_df['CHD'], name='CHD Risk',
                             marker_color='red'),
                      row=row, col=col_num)
    fig.update_layout(height=400 * (len(key_categorical_features)//2 + len(key_categorical_features)%2),
                      title_text="Proportion of CHD Risk by Categorical Features", showlegend=False)
    fig.show()
    logging.info("Plotly: Bar plots showing categorical feature distribution by target generated.")
else:
    logging.warning("No key categorical features identified for target relationship plots.")

logging.info("All EDA visualizations completed.")


## 6. Visual Representation of Correlation, Covariance

Understanding the relationships between features is critical. Correlation measures the linear relationship between two variables, while covariance measures how two variables change together.

*   **Covariance**: A positive covariance indicates that variables tend to move in the same direction, while a negative covariance indicates they tend to move in opposite directions. Its magnitude is not easily interpretable, as it depends on the units of the variables.
*   **Correlation**: A normalized version of covariance, ranging from -1 to 1.
    *   `+1`: Perfect positive linear relationship.
    *   `-1`: Perfect negative linear relationship.
    *   `0`: No linear relationship.
    Correlation is more interpretable as it's unitless.

We will visualize the correlation matrix of our preprocessed dataset to identify strong relationships, especially with our target variable, and potential multicollinearity among features.


In [ ]:
logging.info("Calculating and visualizing correlation and covariance matrices.")

# Calculate the correlation matrix
correlation_matrix = df_processed.corr()

# Calculate the covariance matrix
covariance_matrix = df_processed.cov()

# --- Visualizing Correlation Matrix ---
fig = px.imshow(correlation_matrix,
                text_auto=True,
                aspect="auto",
                color_continuous_scale=px.colors.sequential.RdBu,
                title="Correlation Matrix of All Features")
fig.update_layout(height=800, width=800, title_x=0.5)
fig.show()
logging.info("Plotly: Correlation matrix heatmap generated.")

# --- Visualizing Correlation with Target Variable ---
# Sort by correlation with TenYearCHD
target_correlation = correlation_matrix['TenYearCHD'].sort_values(ascending=False)
target_correlation = target_correlation.drop('TenYearCHD') # Remove self-correlation

fig = px.bar(x=target_correlation.index, y=target_correlation.values,
             title='Feature Correlation with TenYearCHD',
             labels={'x': 'Feature', 'y': 'Correlation Coefficient'},
             color=target_correlation.values,
             color_continuous_scale=px.colors.sequential.RdBu,
             height=500)
fig.update_layout(xaxis_tickangle=-45)
fig.show()
logging.info("Plotly: Bar chart of feature correlation with TenYearCHD generated.")

# --- Visualizing Covariance Matrix (Optional, for completeness. Correlation is usually preferred for interpretation) ---
# Covariance values can be very large or small depending on feature scales, making direct visualization less intuitive than correlation.
# We'll use a heatmap but emphasize that interpretation is tricky due to scale.
fig = px.imshow(covariance_matrix,
                text_auto=False, # Don't auto-add text due to varied magnitudes
                aspect="auto",
                color_continuous_scale=px.colors.sequential.Viridis,
                title="Covariance Matrix of All Features (Magnitude-Dependent)")
fig.update_layout(height=800, width=800, title_x=0.5)
fig.show()
logging.info("Plotly: Covariance matrix heatmap generated.")

logging.info("Correlation and covariance visualizations completed.")


## 7. Feature Selection based on EDA

Based on our EDA and correlation analysis, we will select a subset of features for model training. The criteria for selection include:
*   **Correlation with Target**: Features showing a stronger correlation (positive or negative) with `TenYearCHD` are generally more informative.
*   **Domain Knowledge**: Features known to be strong indicators of cardiovascular health (e.g., age, blood pressure, cholesterol, diabetes, smoking status).
*   **Minimizing Multicollinearity**: Avoiding features that are highly correlated with each other, as this can destabilize some models and make interpretation difficult.
*   **Engineered Features**: Including the newly created `BMI_Category`, `BP_Category`, and `HighCholesterol` if they show promise.

From the correlation analysis, we observe that `age`, `sysBP`, `diaBP`, `glucose`, `diabetes`, `prevalentHyp`, `totChol`, `BMI`, `cigsPerDay`, and `male` generally show higher correlation with `TenYearCHD`. The engineered features `BP_Category`, `HighCholesterol` also appear relevant and might encapsulate risk better than their raw continuous counterparts. `education`, `heartRate`, `currentSmoker`, `BPMeds`, `prevalentStroke` also contribute but might be less impactful individually.

For a first pass, we'll select most features, prioritizing those with higher correlations and including our engineered features, to allow the models to learn complex relationships. We'll be mindful of potential multicollinearity between `sysBP`/`diaBP` and `BP_Category`, and `totChol` and `HighCholesterol`, but keep both for now as they represent different forms of information (continuous vs. categorized risk).


In [ ]:
logging.info("Starting Feature Selection.")

# Define the target variable
target = 'TenYearCHD'

# Features to exclude from training directly (like original ID or target itself)
# 'education' is ordinal and scaled, can be kept. 'prevalentStroke' is a strong indicator, but relatively rare in the dataset, still keep.
# The binary and engineered categorical features are good to keep as they are direct risk factors.
# All numerical features that have been scaled and imputed are potential candidates.

selected_features = [
    'male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds',
    'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP',
    'diaBP', 'BMI', 'heartRate', 'glucose',
    'BMI_Category', 'BP_Category', 'HighCholesterol'
]

# Ensure all selected features are present in the processed DataFrame
selected_features = [f for f in selected_features if f in df_processed.columns]

# Log the selected features
logging.info(f"Selected {len(selected_features)} features for modeling: {selected_features}")

# Separate features (X) and target (y)
X = df_processed[selected_features]
y = df_processed[target]

logging.info(f"Features (X) shape: {X.shape}")
logging.info(f"Target (y) shape: {y.shape}")
logging.info("Feature Selection completed.")


## 8. Separate the Selected Features for Training

After selecting our features, we need to split the dataset into training and testing sets.
*   **Training Set**: Used to train the machine learning models.
*   **Testing Set**: Used to evaluate the performance of the trained models on unseen data. This helps assess how well the model generalizes.

We will use a standard 80/20 split (80% for training, 20% for testing). Stratified splitting will be used to ensure that the proportion of the target variable (`TenYearCHD`) is maintained in both training and testing sets, which is particularly important for imbalanced datasets.

We are taking these selected features because:
*   They showed varying degrees of correlation with the target variable, indicating their potential predictive power.
*   They represent key physiological, demographic, and behavioral risk factors for CHD, aligning with medical understanding.
*   Engineered features (`BMI_Category`, `BP_Category`, `HighCholesterol`) were included as they categorize continuous risk factors into more interpretable and potentially non-linear forms that models can learn from.


In [ ]:
logging.info("Separating selected features into training and testing sets.")

# Split the data into training and testing sets (80% train, 20% test)
# Use 'stratify=y' to maintain the same proportion of target classes in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

logging.info(f"Training set shape (X_train): {X_train.shape}, (y_train): {y_train.shape}")
logging.info(f"Testing set shape (X_test): {X_test.shape}, (y_test): {y_test.shape}")

# Check class distribution in train and test sets
logging.info("Class distribution in training set:")
print(y_train.value_counts(normalize=True))
logging.info("Class distribution in testing set:")
print(y_test.value_counts(normalize=True))

# --- Handling Class Imbalance (SMOTE) ---
# As identified in EDA, the target variable is imbalanced.
# We will apply SMOTE (Synthetic Minority Over-sampling Technique) to the training data only.
# This creates synthetic samples of the minority class to balance the class distribution,
# preventing the model from being biased towards the majority class.
logging.info("Checking for class imbalance in training data and applying SMOTE if necessary.")
if y_train.value_counts()[0] / y_train.value_counts()[1] > 1.5: # Simple ratio check
    logging.info("Applying SMOTE to the training data to handle class imbalance.")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    logging.info(f"X_train shape after SMOTE: {X_train_resampled.shape}")
    logging.info(f"y_train distribution after SMOTE:\n{y_train_resampled.value_counts(normalize=True)}")
    X_train = X_train_resampled
    y_train = y_train_resampled
else:
    logging.info("Class imbalance is not severe enough for SMOTE or has been handled.")

logging.info("Data separation and imbalance handling completed.")


## 9. Modeling

We will now train several classification models to predict `TenYearCHD`. For each model, we will explain its choice and relevance to the problem.

1.  **Logistic Regression**: A linear model used for binary classification. It's chosen for its simplicity, interpretability, and as a strong baseline model. It estimates the probability of an instance belonging to a particular class.
2.  **Decision Tree Classifier**: A non-linear model that partitions the data into subsets based on feature values. It's intuitive and can capture complex relationships, but prone to overfitting.
3.  **Random Forest Classifier**: An ensemble learning method that builds multiple decision trees and merges their predictions. It reduces overfitting compared to single decision trees and generally offers higher accuracy. It's robust and often performs well out-of-the-box.
4.  **Gradient Boosting Classifier (XGBoost/LightGBM/CatBoost)**: Another ensemble method that builds trees sequentially, where each new tree corrects errors made by previous ones. It's known for high performance and efficiency. We'll use `GradientBoostingClassifier` from `sklearn` for simplicity in this context.
5.  **Support Vector Machine (SVC)**: A powerful model that finds an optimal hyperplane to separate classes in a high-dimensional space. It works well with clear margins of separation and is effective in high-dimensional spaces.


In [ ]:
logging.info("Starting Model Training.")

models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced'), # 'balanced' handles imbalance
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'), # 'balanced' handles imbalance
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(random_state=42, probability=True, class_weight='balanced') # 'balanced' handles imbalance, probability=True for ROC AUC
}

trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    logging.info(f"Training {name}...")
    try:
        model.fit(X_train, y_train)
        trained_models[name] = model
        y_pred = model.predict(X_test)
        predictions[name] = y_pred

        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
            probabilities[name] = y_proba
        else: # For SVC without probability=True, this would be an issue, but we set it.
            logging.warning(f"Model {name} does not have predict_proba method (or probability=True not set for SVC). Skipping ROC AUC for this model.")
            probabilities[name] = [0.5] * len(y_test) # Dummy probabilities to avoid errors if needed

        logging.info(f"{name} trained successfully.")
    except Exception as e:
        logging.error(f"Error training {name}: {e}")
        continue

logging.info("Model Training completed.")


## 10. Evaluation Metrics

For a binary classification task like CHD prediction, several metrics are important for a comprehensive evaluation:

*   **Accuracy**: The proportion of correctly classified instances (both true positives and true negatives) out of the total instances.
    *   *Why*: Provides a general sense of how well the model performs, but can be misleading in imbalanced datasets.
*   **Precision (Positive Predictive Value)**: The proportion of true positive predictions among all positive predictions.
    *   *Why*: Relevant when the cost of false positives is high (e.g., unnecessary medical tests due to a false positive CHD prediction).
*   **Recall (Sensitivity, True Positive Rate)**: The proportion of true positive predictions among all actual positive instances.
    *   *Why*: Critical when the cost of false negatives is high (e.g., failing to predict CHD when it exists, leading to delayed treatment).
*   **F1-Score**: The harmonic mean of precision and recall. It provides a single score that balances both precision and recall.
    *   *Why*: Especially useful in imbalanced datasets, as it accounts for both false positives and false negatives.
*   **ROC AUC (Receiver Operating Characteristic Area Under the Curve)**: Measures the ability of the model to distinguish between classes. It's the area under the ROC curve, which plots the True Positive Rate (Recall) against the False Positive Rate at various threshold settings.
    *   *Why*: Provides a robust measure of model performance across all possible classification thresholds and is less sensitive to class imbalance than accuracy.
*   **Confusion Matrix**: A table used to describe the performance of a classification model on a set of test data for which the true values are known. It helps visualize true positives, true negatives, false positives, and false negatives.
    *   *Why*: Provides a detailed breakdown of correct and incorrect classifications.


In [ ]:
logging.info("Starting Model Evaluation.")

evaluation_results = {}

print("\n--- Model Evaluation Results ---")
for name, y_pred in predictions.items():
    logging.info(f"Evaluating {name}...")
    try:
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        metrics = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        }

        # Calculate ROC AUC if probabilities are available
        if name in probabilities and len(probabilities[name]) == len(y_test):
            roc_auc = roc_auc_score(y_test, probabilities[name])
            metrics['ROC-AUC'] = roc_auc
        else:
            metrics['ROC-AUC'] = None
            logging.warning(f"ROC-AUC not calculated for {name} due to missing probabilities.")

        evaluation_results[name] = metrics

        print(f"\n--- {name} ---")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        if 'ROC-AUC' in metrics and metrics['ROC-AUC'] is not None:
            print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                           labels=dict(x="Predicted", y="Actual", color="Count"),
                           x=['No CHD', 'CHD'], y=['No CHD', 'CHD'],
                           title=f'Confusion Matrix for {name}')
        fig_cm.show()

    except Exception as e:
        logging.error(f"Error during evaluation of {name}: {e}")

logging.info("Model Evaluation completed. Detailed results stored in 'evaluation_results'.")


## 11. Local Minima vs. Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of optimization, particularly when training machine learning models, we often deal with an objective function (e.g., a loss function) that we aim to minimize.

*   **Global Minimum**: This is the point in the parameter space where the objective function has the lowest possible value across its entire domain. Achieving the global minimum means finding the absolute best set of parameters for the model, given the data and the model architecture.

*   **Local Minimum**: This is a point where the objective function is lower than all its neighboring points within a certain region, but not necessarily the lowest value across the entire domain. An optimization algorithm might get "stuck" in a local minimum if it doesn't have mechanisms to escape it, leading to a suboptimal model.

Many machine learning algorithms, especially those that rely on iterative optimization like gradient descent (e.g., Logistic Regression, Neural Networks), seek to find the minimum of their loss function. For convex loss functions (like Logistic Regression's log-loss), there's only one global minimum, making optimization straightforward. However, for non-convex loss functions (common in deep learning), multiple local minima can exist, making it challenging to guarantee convergence to the global minimum.

### Visual Representation of Gradient Descent

**Gradient Descent** is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (the steepest ascent) of the function at the current point. The size of these steps is determined by the learning rate.

To illustrate gradient descent conceptually with our dataset, we can visualize the loss surface for a simplified model (e.g., a logistic regression with two features) or just conceptually show the path taken by the algorithm. Directly visualizing the loss surface for a model with many features (as in our case) is impossible in 2D/3D. Instead, we can create a conceptual plot that represents a loss surface and how gradient descent would navigate it.


In [ ]:
logging.info("Visualizing conceptual gradient descent and local vs. global minima.")

# --- Conceptual Visualization of Loss Landscape with Gradient Descent ---
# This is a conceptual illustration, not directly from the multi-dimensional loss of our models.

# Define a 2D non-convex function to simulate a loss landscape with local and global minima
def loss_function(x, y):
    return (np.sin(x/2) + np.cos(y/2)) * (x**2 + y**2) / 20 + 0.5 * (x - 2)**2 + 0.5 * (y + 1)**2 + 10

# Create a grid for the contour plot
x_vals = np.linspace(-10, 10, 100)
y_vals = np.linspace(-10, 10, 100)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_grid = loss_function(X_grid, Y_grid)

# Plot the loss landscape
fig = go.Figure(data=[go.Surface(z=Z_grid, x=x_vals, y=y_vals, colorscale='Viridis', opacity=0.8)])

# Add contour lines for better visibility of minima
fig.add_trace(go.Contour(z=Z_grid, x=x_vals, y=y_vals,
                         colorscale='gray', showscale=False,
                         contours_coloring='lines', line_width=1,
                         opacity=0.5))

# Simulate a gradient descent path
# (This is just an illustrative path, not an actual GD run on a simplified model from our dataset)
# Start point for GD
start_x, start_y = -8, 8
learning_rate = 0.1
num_steps = 50

path_x = [start_x]
path_y = [start_y]
path_z = [loss_function(start_x, start_y)]

# Simplified gradient calculation (conceptual)
# In reality, this would be analytical gradients of the actual loss function.
def gradient(x, y):
    grad_x = 0.2 * x + (x**2 + y**2)/20 * np.cos(x/2)/2 + np.sin(x/2) * x/10 - 2
    grad_y = y + (x**2 + y**2)/20 * (-np.sin(y/2)/2) + np.cos(y/2) * y/10 + 1
    return np.array([grad_x, grad_y])

current_x, current_y = start_x, start_y
for _ in range(num_steps):
    grad_val = gradient(current_x, current_y)
    current_x = current_x - learning_rate * grad_val[0]
    current_y = current_y - learning_rate * grad_val[1]
    path_x.append(current_x)
    path_y.append(current_y)
    path_z.append(loss_function(current_x, current_y))

# Add the gradient descent path to the plot
fig.add_trace(go.Scatter3d(x=path_x, y=path_y, z=path_z,
                           mode='lines+markers',
                           marker=dict(size=4, color='red'),
                           line=dict(color='red', width=3),
                           name='Gradient Descent Path'))

# Mark conceptual local and global minima
fig.add_trace(go.Scatter3d(x=[2], y=[-1], z=[loss_function(2,-1)],
                           mode='markers', marker=dict(size=8, color='green', symbol='star'),
                           name='Conceptual Global Minimum'))
fig.add_trace(go.Scatter3d(x=[-5], y=[-5], z=[loss_function(-5,-5)],
                           mode='markers', marker=dict(size=8, color='orange', symbol='circle'),
                           name='Conceptual Local Minimum'))

fig.update_layout(title='Conceptual Loss Landscape with Gradient Descent Path',
                  scene=dict(
                      xaxis_title='Parameter 1',
                      yaxis_title='Parameter 2',
                      zaxis_title='Loss Function Value'),
                  width=900, height=800,
                  legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)'))
fig.show()
logging.info("Plotly: Conceptual visualization of loss landscape with gradient descent generated.")


## 12. Residuals and How to Visualize Them

### What are Residuals?

In general, a **residual** is the difference between an observed value and a predicted value by a model.
`Residual = Observed Value - Predicted Value`

While residuals are most directly applicable and interpretable in regression tasks (where observed and predicted values are continuous), the concept can be extended to classification. For classification, we don't have continuous predicted values in the same way, but we can consider "residuals" in terms of classification errors:
*   **Misclassifications**: Instances where the predicted class does not match the true class.
*   **Probability Residuals**: For models that output probabilities, we could consider `y_true - y_pred_proba` as a form of residual, though this is less common than in regression.

### How to Visualize Residuals (for Classification)

For binary classification, the most common and effective ways to visualize errors (which are analogous to residuals) are:

1.  **Confusion Matrix**: As shown in the evaluation section, this directly counts True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
    *   **False Positives (FP)**: The model predicted CHD (1) but the patient did not develop it (0). These are instances where the model "over-predicted" the risk.
    *   **False Negatives (FN)**: The model predicted no CHD (0) but the patient did develop it (1). These are instances where the model "under-predicted" the risk, which can be very costly in medical diagnosis.

2.  **ROC Curve**: Plots the True Positive Rate (Recall) against the False Positive Rate (FP / (FP + TN)) at various classification thresholds. It helps assess the model's ability to discriminate between classes across different thresholds.

3.  **Predicted Probabilities vs. True Labels**:
    *   We can visualize the distribution of predicted probabilities for each true class. Ideally, the probabilities for the true `0` class should be concentrated near `0`, and for the true `1` class, concentrated near `1`. Overlap indicates uncertainty or misclassification.

### Comparison and Metrics to Improve Them

Analyzing residuals (misclassifications) helps us understand where the model struggles:

*   **High False Negatives (FN)**: The model is missing many actual positive cases.
    *   *Metrics to improve*: Focus on increasing **Recall**. Techniques include adjusting classification threshold, using class weighting, or oversampling the minority class (e.g., SMOTE).
*   **High False Positives (FP)**: The model is incorrectly predicting positive cases.
    *   *Metrics to improve*: Focus on increasing **Precision**. Techniques include adjusting classification threshold, incorporating more discriminative features, or using models less prone to false alarms.
*   **Both High FP and FN**: The model might be poorly calibrated or lack sufficient predictive power.
    *   *Metrics to improve*: Focus on **F1-Score** or **ROC-AUC**. This often requires more fundamental changes like feature engineering, trying more complex models, or collecting more data.

By visualizing the distributions of features for misclassified points, we might identify patterns or specific subsets of data where the model performs poorly.


In [ ]:
logging.info("Visualizing residuals (misclassifications) for the best model (e.g., Random Forest).")

# Let's pick the Random Forest Classifier for residual analysis as it's often a strong performer.
best_model_name = 'Random Forest' # We'll re-confirm this after hyperparameter tuning
if best_model_name in trained_models:
    model = trained_models[best_model_name]
    y_pred = predictions[best_model_name]
    y_proba = probabilities[best_model_name]

    print(f"\n--- Residual Analysis for {best_model_name} ---")

    # 1. Confusion Matrix (already shown above, but reiterating its importance for residuals)
    cm = confusion_matrix(y_test, y_pred)
    logging.info(f"Confusion Matrix for {best_model_name}:\n{cm}")
    print("Interpretation of Confusion Matrix for residuals:")
    print(f"  True Negatives (TN): {cm[0,0]} - Correctly predicted no CHD.")
    print(f"  False Positives (FP): {cm[0,1]} - Predicted CHD, but actually no CHD (Type I error). These are the 'residuals' where the model overestimated risk.")
    print(f"  False Negatives (FN): {cm[1,0]} - Predicted no CHD, but actually CHD (Type II error). These are the 'residuals' where the model underestimated risk.")
    print(f"  True Positives (TP): {cm[1,1]} - Correctly predicted CHD.")

    # 2. ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    fig_roc = go.Figure()
    fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                 name=f'{best_model_name} (AUC = {roc_auc:.4f})'))
    fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier',
                                 line=dict(dash='dash', color='gray')))
    fig_roc.update_layout(title=f'ROC Curve for {best_model_name}',
                          xaxis_title='False Positive Rate',
                          yaxis_title='True Positive Rate',
                          width=600, height=500)
    fig_roc.show()
    logging.info(f"Plotly: ROC curve for {best_model_name} generated.")

    # 3. Distribution of Predicted Probabilities by True Label
    df_proba = pd.DataFrame({'True Label': y_test, 'Predicted Probability': y_proba})
    df_proba['True Label'] = df_proba['True Label'].map({0: 'No CHD', 1: 'CHD'})

    fig_proba = px.histogram(df_proba, x='Predicted Probability', color='True Label',
                             facet_col='True Label', histnorm='percent',
                             title=f'Predicted Probability Distribution for {best_model_name} by True Label',
                             labels={'True Label': 'Actual Outcome'},
                             color_discrete_map={'No CHD': 'blue', 'CHD': 'red'})
    fig_proba.update_layout(bargap=0.1)
    fig_proba.show()
    logging.info(f"Plotly: Predicted probability distribution for {best_model_name} generated.")

    # How to improve:
    logging.info("\nInsights for improving model based on residuals:")
    if cm[1,0] > cm[0,1]: # More False Negatives than False Positives
        logging.info("The model has a higher number of False Negatives (missing actual CHD cases).")
        logging.info("Focus on increasing Recall. Consider lowering the classification threshold (if appropriate for the application), applying stronger class weighting for the minority class, or using more advanced oversampling techniques.")
    elif cm[0,1] > cm[1,0]: # More False Positives than False Negatives
        logging.info("The model has a higher number of False Positives (incorrectly predicting CHD).")
        logging.info("Focus on increasing Precision. Consider raising the classification threshold (if appropriate), incorporating more robust features, or using models that emphasize precision.")
    else:
        logging.info("False Positives and False Negatives are relatively balanced. Focus on overall F1-Score and ROC-AUC improvement.")
        logging.info("Further feature engineering, more complex models, or collecting more data might be beneficial.")

else:
    logging.warning(f"Model '{best_model_name}' not found in trained_models for residual analysis.")

logging.info("Residual analysis completed.")


## 13. Overfitting or Underfitting

### What are Overfitting and Underfitting?

*   **Overfitting**: Occurs when a model learns the training data too well, including the noise and outliers, to the extent that it performs poorly on unseen data. An overfitted model has high variance and low bias.
    *   **Detection**: High accuracy on the training set but significantly lower accuracy on the testing/validation set.
    *   **Analogy**: A student who memorizes answers to practice questions but doesn't understand the concepts, failing the actual exam.

*   **Underfitting**: Occurs when a model is too simple to capture the underlying patterns in the data. It performs poorly on both training and testing sets. An underfitted model has high bias and low variance.
    *   **Detection**: Low accuracy on both the training and testing sets.
    *   **Analogy**: A student who doesn't study enough or is given a trivial curriculum, failing to learn even basic concepts.

### How to Fix Overfitting:
1.  **More Data**: The simplest solution; more data helps the model learn the true underlying patterns rather than noise.
2.  **Feature Selection**: Remove irrelevant or redundant features that might be contributing to noise learning.
3.  **Feature Engineering**: Create more meaningful features that capture the true signal in the data.
4.  **Regularization**: Add a penalty term to the loss function to discourage overly complex models (e.g., L1, L2 regularization in Logistic Regression, Ridge, Lasso).
5.  **Simpler Models**: Use models with fewer parameters or less complexity (e.g., Logistic Regression instead of a deep neural network).
6.  **Ensemble Methods**: Bagging methods like Random Forest inherently reduce variance and overfitting by averaging predictions from multiple models.
7.  **Cross-Validation**: Use techniques like K-Fold cross-validation to get a more robust estimate of model performance and detect overfitting early.
8.  **Pruning (Decision Trees)**: Limit the depth or number of leaves in decision trees.
9.  **Dropout (Neural Networks)**: Randomly drop units during training to prevent complex co-adaptations.

### How to Fix Underfitting:
1.  **More Complex Model**: Use a model with more parameters or higher capacity (e.g., add more layers to a neural network, use a non-linear kernel for SVM).
2.  **More Features**: Introduce more relevant features or use feature engineering to create more informative ones.
3.  **Reduce Regularization**: If regularization was applied, reducing its strength can allow the model to learn more complex patterns.
4.  **Increase Training Time/Epochs**: For iterative models, train for more iterations, provided it doesn't lead to overfitting.
5.  **Remove Noise**: Clean the data by handling outliers and errors, which can confuse a simpler model.

### Detecting Overfitting/Underfitting in our Models:

We can check for these by comparing the training set scores with the test set scores.


In [ ]:
logging.info("Checking for overfitting/underfitting by comparing train and test scores.")

train_scores = {}
test_scores = {}

print("\n--- Overfitting/Underfitting Analysis ---")
for name, model in trained_models.items():
    logging.info(f"Analyzing {name} for overfitting/underfitting.")
    try:
        y_train_pred = model.predict(X_train)
        y_test_pred = predictions[name] # Already calculated

        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)

        train_scores[name] = train_accuracy
        test_scores[name] = test_accuracy

        print(f"\nModel: {name}")
        print(f"  Training Accuracy: {train_accuracy:.4f}")
        print(f"  Testing Accuracy:  {test_accuracy:.4f}")

        if train_accuracy > test_accuracy + 0.1: # A heuristic threshold for significant difference
            logging.warning(f"{name} might be overfitting. Training score is significantly higher than testing score.")
            print(f"  Observation: Likely **Overfitting** (Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}).")
            print("  Potential fixes: Consider regularization, pruning (for trees), more data, or simpler features.")
        elif train_accuracy < 0.6 and test_accuracy < 0.6: # Another heuristic for poor performance
            logging.warning(f"{name} might be underfitting. Both train and test scores are low.")
            print(f"  Observation: Likely **Underfitting** (Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}).")
            print("  Potential fixes: Use a more complex model, engineer more features, or reduce regularization.")
        else:
            logging.info(f"{name} shows reasonable fit. Training and testing scores are comparable.")
            print(f"  Observation: Good Fit. Training and testing scores are comparable.")
    except Exception as e:
        logging.error(f"Error during overfitting/underfitting analysis for {name}: {e}")

logging.info("Overfitting/underfitting analysis completed.")


## 14. Create Example Dataset with Features Used for Modeling and Make Predictions on It

To demonstrate the usability of our trained models, we'll create a small, new dataset with hypothetical patient data. This dataset will have the same features as those used for training and will undergo the same preprocessing steps (scaling, specifically) before making predictions.


In [ ]:
logging.info("Creating example dataset for prediction.")

# Define a sample data point based on our original feature schema and types
# The values here are hypothetical for a new individual.
# The order of columns should match the `selected_features`
example_data = {
    'male': [1, 0],  # Male, Female
    'age': [55, 40], # Age 55, Age 40
    'education': [2.0, 4.0], # High school, College grad
    'currentSmoker': [1, 0], # Smoker, Non-smoker
    'cigsPerDay': [20.0, 0.0], # 20 cigarettes/day, 0 cigarettes/day
    'BPMeds': [0.0, 0.0], # No BP meds
    'prevalentStroke': [0, 0], # No stroke history
    'prevalentHyp': [1, 0], # Hypertensive, Not hypertensive
    'diabetes': [0, 0], # No diabetes
    'totChol': [280.0, 190.0], # High cholesterol, Normal cholesterol
    'sysBP': [150.0, 115.0], # High systolic BP, Normal systolic BP
    'diaBP': [90.0, 75.0], # High diastolic BP, Normal diastolic BP
    'BMI': [32.0, 22.0], # Obese, Normal BMI
    'heartRate': [80.0, 70.0], # Normal heart rate
    'glucose': [110.0, 85.0] # Elevated glucose, Normal glucose
}

# Create a DataFrame from the example data
new_patients_df = pd.DataFrame(example_data)
logging.info("Raw example data created.")
print("Raw example data:")
print(new_patients_df)

# --- Apply Preprocessing Steps to Example Data ---
# 1. Feature Engineering (must be applied first, as it creates new features from raw ones)
def get_bmi_category_new(bmi):
    if bmi < 18.5: return 0
    elif 18.5 <= bmi < 25: return 1
    elif 25 <= bmi < 30: return 2
    else: return 3
new_patients_df['BMI_Category'] = new_patients_df['BMI'].apply(get_bmi_category_new)

def get_bp_category_new(sysBP, diaBP):
    if sysBP < 120 and diaBP < 80: return 0
    elif (120 <= sysBP <= 129) and diaBP < 80: return 1
    elif (130 <= sysBP <= 139) or (80 <= diaBP <= 89): return 2
    elif (sysBP >= 140) or (diaBP >= 90): return 3
    else: return 0
new_patients_df['BP_Category'] = new_patients_df.apply(lambda row: get_bp_category_new(row['sysBP'], row['diaBP']), axis=1)

new_patients_df['HighCholesterol'] = (new_patients_df['totChol'] >= 240).astype(int)
logging.info("Feature engineering applied to example data.")

# Ensure the example data has the same columns as X_train, in the same order
# and only include the selected features
new_patients_processed = new_patients_df[selected_features]

# 2. Scaling (using the SAME scaler fitted on the training data)
# Identify columns that were scaled in the original training data
features_to_scale_example = [col for col in continuous_cols + ['education'] if col in new_patients_processed.columns]

try:
    new_patients_processed[features_to_scale_example] = scaler.transform(new_patients_processed[features_to_scale_example])
    logging.info("Example data scaled using the previously fitted StandardScaler.")
except Exception as e:
    logging.error(f"Error during scaling example data: {e}")

logging.info("Processed example data for prediction:")
print(new_patients_processed)

# Make predictions using the best model (Random Forest, for now)
# We will use the model that performed best after hyperparameter tuning as the "final" model.
# For now, let's pick one of the base models, e.g., Random Forest.
model_for_prediction = trained_models.get('Random Forest') # Get the Random Forest model

if model_for_prediction:
    try:
        predictions_example = model_for_prediction.predict(new_patients_processed)
        probabilities_example = model_for_prediction.predict_proba(new_patients_processed)[:, 1]

        print("\n--- Predictions on Example Dataset (using Random Forest) ---")
        for i in range(len(new_patients_df)):
            print(f"Patient {i+1}:")
            print(f"  Features: {new_patients_df.iloc[i].to_dict()}")
            print(f"  Predicted TenYearCHD: {'Yes' if predictions_example[i] == 1 else 'No'} (Probability: {probabilities_example[i]:.4f})")
            logging.info(f"Prediction for example patient {i+1}: CHD={predictions_example[i]}, Proba={probabilities_example[i]:.4f}")
    except Exception as e:
        logging.error(f"Error making predictions on example data: {e}")
else:
    logging.error("Random Forest model not found for example prediction. Please ensure it was trained successfully.")

logging.info("Example dataset prediction completed.")


## 15. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is the process of finding the best set of hyperparameters for a machine learning model that results in the optimal performance. We'll use `GridSearchCV` to systematically search through a predefined set of hyperparameter values for one of our promising models, the Random Forest Classifier.

For demonstration purposes and to manage computation time, we'll perform tuning on a moderately sized grid for Random Forest.

**Random Forest Classifier Hyperparameters for Tuning:**
*   `n_estimators`: The number of trees in the forest. More trees generally lead to better performance but increase computation.
*   `max_depth`: The maximum depth of the tree. Limits the number of splits, controlling overfitting.
*   `min_samples_split`: The minimum number of samples required to split an internal node.
*   `min_samples_leaf`: The minimum number of samples required to be at a leaf node.
*   `criterion`: The function to measure the quality of a split (e.g., 'gini' for Gini impurity, 'entropy' for information gain).
*   `class_weight`: Handles class imbalance (already set to 'balanced').

We'll use `f1_weighted` as the scoring metric because our dataset is imbalanced and F1-score provides a good balance between precision and recall.


In [ ]:
logging.info("Starting Hyperparameter Tuning for Random Forest Classifier.")

# Define the model to tune
model = RandomForestClassifier(random_state=42, class_weight='balanced')

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None], # None means nodes are expanded until all leaves are pure or contain less than min_samples_split samples.
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'criterion': ['gini', 'entropy']
}

# Initialize GridSearchCV
# Using `f1_weighted` as the scoring metric for imbalanced datasets
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=2)

try:
    logging.info("Fitting GridSearchCV (this may take a while)...")
    grid_search.fit(X_train, y_train) # Fit on the resampled training data

    logging.info("Hyperparameter tuning completed.")

    # Get the best parameters and best score
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    best_estimator = grid_search.best_estimator_

    logging.info(f"Best hyperparameters found: {best_params}")
    logging.info(f"Best cross-validation F1-weighted score: {best_score:.4f}")

    print("\n--- Hyperparameter Tuning Results for Random Forest ---")
    print(f"Best Parameters: {best_params}")
    print(f"Best F1-weighted Score: {best_score:.4f}")

    # Evaluate the best estimator on the test set
    y_pred_tuned = best_estimator.predict(X_test)
    y_proba_tuned = best_estimator.predict_proba(X_test)[:, 1]

    tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
    tuned_precision = precision_score(y_test, y_pred_tuned)
    tuned_recall = recall_score(y_test, y_pred_tuned)
    tuned_f1 = f1_score(y_test, y_pred_tuned)
    tuned_roc_auc = roc_auc_score(y_test, y_proba_tuned)

    logging.info(f"Tuned Random Forest - Test Accuracy: {tuned_accuracy:.4f}, Precision: {tuned_precision:.4f}, Recall: {tuned_recall:.4f}, F1-Score: {tuned_f1:.4f}, ROC-AUC: {tuned_roc_auc:.4f}")

    print("\nPerformance of Tuned Random Forest on Test Set:")
    print(f"Accuracy: {tuned_accuracy:.4f}")
    print(f"Precision: {tuned_precision:.4f}")
    print(f"Recall: {tuned_recall:.4f}")
    print(f"F1-Score: {tuned_f1:.4f}")
    print(f"ROC-AUC: {tuned_roc_auc:.4f}")
    print("\nClassification Report (Tuned Random Forest):")
    print(classification_report(y_test, y_pred_tuned))

    # Update the models dictionary with the tuned model
    trained_models['Random Forest (Tuned)'] = best_estimator
    predictions['Random Forest (Tuned)'] = y_pred_tuned
    probabilities['Random Forest (Tuned)'] = y_proba_tuned
    evaluation_results['Random Forest (Tuned)'] = {
        'Accuracy': tuned_accuracy, 'Precision': tuned_precision,
        'Recall': tuned_recall, 'F1-Score': tuned_f1, 'ROC-AUC': tuned_roc_auc
    }

except Exception as e:
    logging.error(f"Error during hyperparameter tuning: {e}")
    best_estimator = trained_models.get('Random Forest') # Fallback to untuned model if tuning fails
    logging.warning("Falling back to untuned Random Forest model for subsequent steps due to tuning error.")

logging.info("Hyperparameter tuning completed.")


## 16. Visual Representation of the Results

Visualizing the results helps in comparing model performance and understanding the types of errors made. We'll focus on:
*   **Comparison of Evaluation Metrics**: Bar plots to compare accuracy, precision, recall, F1-score, and ROC-AUC across all models.
*   **ROC Curves**: Plotting ROC curves for all models on the same graph to visually compare their discriminative power.
*   **Confusion Matrices**: Detailed breakdown of TP, TN, FP, FN for the best performing models.


In [ ]:
logging.info("Generating visual representations of model results.")

# --- 1. Comparison of Evaluation Metrics ---
metrics_df = pd.DataFrame(evaluation_results).T
metrics_df = metrics_df.drop(columns=['ROC-AUC'], errors='ignore') # ROC-AUC will be plotted separately

fig_metrics = go.Figure()

for metric in metrics_df.columns:
    fig_metrics.add_trace(go.Bar(x=metrics_df.index, y=metrics_df[metric], name=metric))

fig_metrics.update_layout(title_text='Model Performance Comparison',
                          xaxis_title='Model',
                          yaxis_title='Score',
                          barmode='group',
                          height=600, width=800)
fig_metrics.show()
logging.info("Plotly: Model performance comparison bar chart generated.")

# --- 2. ROC Curves for All Models ---
fig_roc_all = go.Figure()
fig_roc_all.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier',
                                 line=dict(dash='dash', color='gray')))

for name, model_probs in probabilities.items():
    if model_probs is not None and len(model_probs) == len(y_test):
        fpr, tpr, _ = roc_curve(y_test, model_probs)
        roc_auc = roc_auc_score(y_test, model_probs)
        fig_roc_all.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                         name=f'{name} (AUC = {roc_auc:.4f})'))
    else:
        logging.warning(f"Skipping ROC curve for {name} due to invalid probabilities.")

fig_roc_all.update_layout(title='ROC Curves for All Models',
                          xaxis_title='False Positive Rate',
                          yaxis_title='True Positive Rate',
                          width=700, height=600)
fig_roc_all.show()
logging.info("Plotly: ROC curves for all models generated.")

# --- 3. Confusion Matrix for Best Model (Tuned Random Forest) ---
# Assuming 'Random Forest (Tuned)' is the best and exists
best_model_name = 'Random Forest (Tuned)'
if best_model_name not in predictions:
    # Fallback to the best untuned model if tuned version not available
    best_model_name = max(evaluation_results, key=lambda k: evaluation_results[k].get('F1-Score', 0))
    logging.warning(f"Tuned Random Forest not found, using {best_model_name} for final confusion matrix.")

if best_model_name in predictions:
    final_y_pred = predictions[best_model_name]
    final_cm = confusion_matrix(y_test, final_y_pred)
    fig_final_cm = px.imshow(final_cm, text_auto=True, color_continuous_scale='Blues',
                               labels=dict(x="Predicted", y="Actual", color="Count"),
                               x=['No CHD', 'CHD'], y=['No CHD', 'CHD'],
                               title=f'Final Confusion Matrix for {best_model_name}')
    fig_final_cm.show()
    logging.info(f"Plotly: Final Confusion Matrix for {best_model_name} generated.")
else:
    logging.error("No suitable model found for final confusion matrix visualization.")


logging.info("Result visualizations completed.")


## 17. Final Model Selection based on Best Result

After evaluating all models and performing hyperparameter tuning, we need to select the best model. Our primary focus for this medical prediction task, where missing a positive case (False Negative) can have severe consequences, would be a good balance of **Recall** and **Precision**, reflected by the **F1-Score**, and overall discriminative power measured by **ROC-AUC**.

Based on the evaluation results and the visualizations:
*   We look for the model with the highest F1-Score and ROC-AUC on the test set.
*   We also consider the balance between training and testing scores to ensure the model generalizes well and is not severely overfit.

Let's identify the best model based on F1-Score (weighted due to imbalance) and ROC-AUC.


In [ ]:
logging.info("Selecting the final model.")

# Determine the best model based on F1-Score and ROC-AUC
best_f1_score = -1
best_roc_auc = -1
best_model_name_f1 = None
best_model_name_roc_auc = None

for name, metrics in evaluation_results.items():
    if metrics.get('F1-Score', 0) > best_f1_score:
        best_f1_score = metrics['F1-Score']
        best_model_name_f1 = name
    if metrics.get('ROC-AUC', 0) is not None and metrics['ROC-AUC'] > best_roc_auc:
        best_roc_auc = metrics['ROC-AUC']
        best_model_name_roc_auc = name

# If tuned model exists and performs well, it's usually preferred.
# Prioritize the 'Random Forest (Tuned)' if it's available and performs competitively.
if 'Random Forest (Tuned)' in evaluation_results and \
   evaluation_results['Random Forest (Tuned)']['F1-Score'] >= best_f1_score * 0.95: # Allow for slight drop if more robust
    final_model_name = 'Random Forest (Tuned)'
    logging.info("Selecting 'Random Forest (Tuned)' as the final model due to tuning and performance.")
else:
    # If the untuned models are significantly better or tuned RF doesn't exist
    if best_model_name_f1 == best_model_name_roc_auc:
        final_model_name = best_model_name_f1
        logging.info(f"Selecting '{final_model_name}' as the final model based on consistent best F1-Score and ROC-AUC.")
    else:
        # If there's a tie or different bests, choose based on F1-Score as it balances precision/recall
        final_model_name = best_model_name_f1
        logging.info(f"Selecting '{final_model_name}' as the final model (best F1-Score, ROC-AUC was similar or slightly lower).")

final_model = trained_models[final_model_name]
final_model_metrics = evaluation_results[final_model_name]

print(f"\n--- Final Model Selected ---")
print(f"Model: {final_model_name}")
print(f"Test Accuracy: {final_model_metrics['Accuracy']:.4f}")
print(f"Test Precision: {final_model_metrics['Precision']:.4f}")
print(f"Test Recall: {final_model_metrics['Recall']:.4f}")
print(f"Test F1-Score: {final_model_metrics['F1-Score']:.4f}")
if final_model_metrics['ROC-AUC'] is not None:
    print(f"Test ROC-AUC: {final_model_metrics['ROC-AUC']:.4f}")

logging.info(f"Final model selected: {final_model_name}. Its test metrics: {final_model_metrics}")
logging.info("Final model selection completed.")


## 18. Ensure to Save the Final Model

It's crucial to save the trained model so it can be deployed and used for future predictions without retraining. We will use the `pickle` library for this purpose and store the model in an `artifacts` directory. It's also good practice to save the `StandardScaler` used for preprocessing, as new data will need to be scaled using the same transformation.


In [ ]:
logging.info("Saving the final model and scaler to artifacts directory.")

# Create the artifacts directory if it doesn't exist (already done at the beginning)
# os.makedirs(artifacts_dir, exist_ok=True)

# Define file paths for the model and scaler
model_filename = os.path.join(artifacts_dir, f'{final_model_name.replace(" ", "_").lower()}_chd_prediction_model.pkl')
scaler_filename = os.path.join(artifacts_dir, 'standard_scaler.pkl')
features_filename = os.path.join(artifacts_dir, 'selected_features.pkl')

try:
    # Save the final model
    with open(model_filename, 'wb') as file:
        pickle.dump(final_model, file)
    logging.info(f"Final model '{final_model_name}' saved successfully to {model_filename}")

    # Save the scaler
    with open(scaler_filename, 'wb') as file:
        pickle.dump(scaler, file)
    logging.info(f"StandardScaler saved successfully to {scaler_filename}")

    # Save the list of selected features to ensure consistency during future predictions
    with open(features_filename, 'wb') as file:
        pickle.dump(selected_features, file)
    logging.info(f"Selected features list saved successfully to {features_filename}")

except Exception as e:
    logging.error(f"Error saving model or scaler: {e}")

logging.info("Model and scaler saving completed.")


## 19. Insights

Based on our comprehensive analysis, here are some key insights:

*   **Data Quality**: The dataset had missing values, particularly in `education`, `cigsPerDay`, `totChol`, `BMI`, `heartRate`, and `glucose`. These were successfully imputed, predominantly using median values for robustness. Outliers were also capped, which is important for models sensitive to extreme values.
*   **Feature Importance**:
    *   **Age** is consistently one of the strongest predictors of CHD, showing a clear positive correlation. This aligns with medical understanding.
    *   **Blood Pressure** (both `sysBP`, `diaBP`, and the engineered `BP_Category`) and **Cholesterol** (`totChol`, `HighCholesterol`) are highly influential factors.
    *   **Diabetes** and **Prevalent Hypertension** are also strong indicators.
    *   **Smoking status** (`cigsPerDay`, `currentSmoker`) and **BMI** (`BMI`, `BMI_Category`) contribute significantly to the risk assessment.
*   **Class Imbalance**: The target variable `TenYearCHD` was imbalanced (minority class was ~15-16%). This was addressed using `SMOTE` on the training data and `class_weight='balanced'` in models, which helped models better learn from the minority class.
*   **Model Performance**:
    *   **Ensemble models** like Random Forest and Gradient Boosting generally outperformed simpler models like Logistic Regression and Decision Trees, especially after hyperparameter tuning. This suggests that the relationship between features and CHD risk is complex and non-linear.
    *   The **Tuned Random Forest Classifier** emerged as the best-performing model, achieving a good balance between precision and recall (F1-Score) and strong discriminative power (ROC-AUC). This indicates its robustness and ability to generalize to unseen data.
*   **Risk Factors**: The model reinforces known medical risk factors for CHD, such as older age, higher blood pressure, elevated cholesterol, smoking, obesity, and diabetes.
*   **Feature Engineering**: Creating categorical features like `BMI_Category`, `BP_Category`, and `HighCholesterol` likely improved model performance by introducing non-linearity and simplifying risk interpretation.


## 20. Conclusion

This project successfully developed and evaluated machine learning models for predicting the 10-year risk of Coronary Heart Disease using the Framingham Heart Study dataset. Through a structured pipeline involving data loading, comprehensive EDA, meticulous preprocessing (missing value imputation, outlier handling, feature scaling, and feature engineering), model training, hyperparameter tuning, and robust evaluation, we identified a high-performing model.

The **Tuned Random Forest Classifier** was selected as the final model due to its superior performance in terms of F1-Score and ROC-AUC, demonstrating its effectiveness in handling the imbalanced nature of the dataset and capturing complex patterns. This model, along with the scaler and selected features, has been saved for future deployment.

**Future Work and Recommendations:**
1.  **More Advanced Feature Engineering**: Explore interaction terms between features (e.g., age * smoking status) or more complex clinical scores.
2.  **Advanced Imputation**: Investigate more sophisticated imputation techniques like MICE (Multiple Imputation by Chained Equations) to potentially preserve more data variance.
3.  **Explore Other Models**: Experiment with deep learning models (e.g., Neural Networks) or other boosting algorithms like LightGBM or CatBoost for potentially higher performance.
4.  **Explainable AI (XAI)**: Implement techniques like SHAP or LIME to provide more interpretable insights into feature contributions for individual predictions, which is crucial in healthcare.
5.  **Longitudinal Data Analysis**: The Framingham study is longitudinal; incorporating time-series aspects of patient data could lead to more robust predictions if such data were available in a suitable format.
6.  **External Validation**: Validate the model's performance on external, independent datasets to ensure generalizability beyond the Framingham cohort.

This machine learning pipeline provides a solid foundation for CHD risk prediction, offering valuable insights that can aid healthcare professionals in identifying high-risk individuals and guiding preventive strategies.